# Notebook 01 — Anatomy of a Scientific Paper

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 1 of 12  
**Time estimate:** 60 minutes

> Before you can write a paper, you must be able to read one at three speeds:
> skim, read, and reconstruct. This notebook dissects the IMRaD structure,
> maps the logical argument flow, and gives you a systematic protocol for
> extracting information from a paper without reading every word.

---
## Step 1 — Motivation

You will read hundreds of papers during a PhD. A researcher who reads every word
of every paper will drown. A researcher who reads only abstracts will miss the
evidence. The skill is strategic reading: extracting what you need at the depth
you need it, efficiently. The same structure — IMRaD — governs what you write.
Understanding it as a reader makes you a better writer.

---
## Step 2 — The IMRaD Structure

**IMRaD = Introduction, Methods, Results, and Discussion.**
Every empirical scientific paper is built on this skeleton — whether it's a
genome assembly, a clinical trial, or a machine learning benchmark.

| Section | Question it answers | What to look for |
|---------|--------------------|-----------------|
| **Abstract** | What is this paper about in 200 words? | Motivation, gap, approach, key result, significance |
| **Introduction** | Why should I care? What is the gap? | The research question — usually in the last paragraph |
| **Methods** | How was it done? Could I repeat it? | Parameters, software, datasets, statistical tests |
| **Results** | What did they find? | Quantitative claims, figure references, comparison to baseline |
| **Discussion** | What does it mean? What are the limits? | Interpretation, comparison to prior work, limitations |
| **Conclusion** | One paragraph summary | Often just the abstract re-stated — skip if pressed for time |

**The C-C-C scheme (Mensh & Kording 2017):** at every level — sentence, paragraph,
section, paper — the structure is **Context → Content → Conclusion**.
- *Context:* where we are, what we know
- *Content:* the new thing
- *Conclusion:* what it means

A sentence that starts with the new thing, without context, forces the reader to
do the work of locating the claim. A sentence that ends with the context, burying
the conclusion, loses the reader's attention.

---
## Step 3 — Three Reading Speeds

**Speed 1 — Skim (3–5 minutes): Is this worth reading?**
1. Title: what is the claim?
2. Abstract: what did they find? What method?
3. Figures + captions: what data do they show?
4. Conclusion paragraph: what do they claim?
Decision: read further, file for later, or skip.

**Speed 2 — Read (30–60 minutes): What did they do and claim?**
Read in order. Mark every claim you don't yet believe with a "?". Mark every
unfamiliar term. Look at every figure. Note: what is the evidence for each claim?

**Speed 3 — Reconstruct (1–2 hours): Do I understand it?**
Close the paper. Re-derive or re-explain the central method in plain language.
Reproduce a key figure (approximately) from the text. Write 3 sentences:
what they did, what they found, what you're not sure about.
This is the Pass-3 from the project constitution (§8). Log it in `paper-notes/`.

**Key diagnostic:** if you can't explain what the paper did in one sentence
without jargon, you haven't read it — you've skimmed it with extra steps.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ---- Computational component: analyse paper structure programmatically ----
# We use a synthetic example abstract to demonstrate text analysis

SYNTHETIC_ABSTRACT = """
The identification of transcription factor (TF) binding sites in genomic sequences
is a central problem in computational biology. Existing methods based on position
weight matrices (PWMs) treat nucleotide positions as independent, failing to capture
interactions between adjacent residues. Here, we present DeepBind-v2, a convolutional
neural network that learns sequence features from ChIP-seq data without relying on
pre-specified motif models. Applied to 690 TF datasets from ENCODE, DeepBind-v2
achieves a mean AUROC of 0.924, outperforming PWM-based methods by 12.3 percentage
points. Analysis of learned convolutional filters reveals biologically interpretable
motifs that are consistent with known TF binding preferences. DeepBind-v2 is
freely available at https://github.com/example/deepbind-v2 and provides a scalable
approach for TF binding prediction across the human genome.
""".strip()

IMRAD_COMPONENTS = {
    'Motivation / field context': [
        'transcription factor', 'binding sites', 'central problem',
        'computational biology'
    ],
    'Gap / problem with existing work': [
        'existing methods', 'failing to capture', 'without relying on'
    ],
    'Approach / what we did': [
        'we present', 'convolutional neural network', 'learns'
    ],
    'Key result (quantitative)': [
        'AUROC', 'outperforming', 'percentage points'
    ],
    'Significance / availability': [
        'freely available', 'scalable', 'human genome'
    ]
}

def find_component_sentences(abstract, components):
    """Identify which sentence best corresponds to each abstract component."""
    sentences = [s.strip() for s in abstract.split('.') if len(s.strip()) > 10]
    results = {}
    for component, keywords in components.items():
        scores = []
        for sent in sentences:
            score = sum(1 for kw in keywords if kw.lower() in sent.lower())
            scores.append(score)
        best_idx = int(np.argmax(scores))
        results[component] = sentences[best_idx]
    return results, sentences

results, sentences = find_component_sentences(SYNTHETIC_ABSTRACT, IMRAD_COMPONENTS)

print('=== Abstract component analysis ===\n')
for component, sentence in results.items():
    print(f'[{component}]')
    print(f'  "{sentence[:90]}..."' if len(sentence) > 90 else f'  "{sentence}"')
    print()
print(f'Total sentences in abstract: {len(sentences)}')
print(f'Word count: {len(SYNTHETIC_ABSTRACT.split())}')

In [ ]:
# ---- Visualize: argument flow diagram + word frequency ----
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: IMRaD paper structure diagram
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
sections = [
    (1.5, 9.0, 7, 0.8, 'ABSTRACT', '#4e79a7', 'What: 200 words, all 5 components'),
    (2.0, 7.7, 6, 0.9, 'INTRODUCTION', '#f28e2b', 'Why: field → gap → question'),
    (1.5, 6.3, 7, 1.1, 'METHODS', '#59a14f', 'How: enough detail to replicate'),
    (1.5, 5.0, 7, 1.1, 'RESULTS', '#e15759', 'What found: figures, quantitative claims'),
    (2.0, 3.7, 6, 1.0, 'DISCUSSION', '#76b7b2', 'What it means: limits, context'),
    (3.0, 2.5, 4, 0.8, 'CONCLUSION', '#edc948', 'Summary: often = abstract'),
]
for x, y, w, h, title, color, desc in sections:
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                           facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(x + w/2, y + h/2 + 0.1, title, ha='center', va='center',
              fontweight='bold', fontsize=10)
    ax.text(x + w/2, y + h/2 - 0.25, desc, ha='center', va='center', fontsize=7.5,
              color='#333333')
ax.set_title('A. IMRaD paper structure\n(argument flows top to bottom)', fontsize=11)

# Panel B: C-C-C scheme at three levels
ax = axes[1]
ax.axis('off')
levels = [
    (0.85, 'PAPER LEVEL', 'Context: field background\nContent: new method/finding\nConclusion: significance'),
    (0.52, 'SECTION LEVEL', 'Context: what we knew\nContent: this experiment\nConclusion: what it shows'),
    (0.20, 'SENTENCE LEVEL', 'Context: known fact\nContent: new info\nConclusion: implication'),
]
colors_ccc = ['#4e79a7', '#f28e2b', '#59a14f']
for (y, title, text), color in zip(levels, colors_ccc):
    ax.text(0.5, y, f'{title}\n{text}', ha='center', va='center', fontsize=9,
              bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3,
                          edgecolor=color),
              transform=ax.transAxes)
ax.set_title('B. C-C-C scheme\n(Context → Content → Conclusion at every level)', fontsize=11)

# Panel C: Abstract word frequency
ax = axes[2]
words = re.findall(r'\b[a-zA-Z]{4,}\b', SYNTHETIC_ABSTRACT.lower())
stopwords = {'that', 'with', 'from', 'this', 'they', 'their', 'have', 'which',
               'based', 'been', 'more', 'than', 'across', 'approach', 'each',
               'between', 'without', 'learned'}
words_filtered = [w for w in words if w not in stopwords]
word_counts = Counter(words_filtered).most_common(15)
terms, counts = zip(*word_counts)
ypos = range(len(terms))
ax.barh(list(ypos), list(counts), color='steelblue', edgecolor='black', alpha=0.8)
ax.set_yticks(list(ypos)); ax.set_yticklabels(list(terms), fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('C. Top words in abstract\n(domain terms should dominate)', fontsize=11)

plt.suptitle('Module 18 NB01: Anatomy of a Scientific Paper', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('paper_anatomy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4 — Common Structural Failures

**1. The buried hypothesis:** the research question appears in paragraph 4 of the
introduction, after three paragraphs of background. Fix: move it to the end of
paragraph 1 as a one-sentence question, then use paragraphs 2–4 to justify it.

**2. Methods masquerading as Results:** "We ran HISAT2 with --dta flag, then
used StringTie to assemble transcripts" belongs in Methods. Results says what
was found, not what was run.

**3. The un-quantified Discussion:** "The model performed well" is not a result.
"The model achieved AUROC 0.87 (95% CI: 0.84–0.90), outperforming the baseline
by 9 percentage points" is.

**4. The overclaiming Conclusion:** "Our results suggest..." is appropriate.
"Our results prove that..." in a paper with N=50 mice is not.

**5. The floating figure:** a figure that is not explicitly cited in the text,
with a vague caption ("Figure 3. Experimental results"). Every figure must be
cited in the text; the caption must stand alone.

---
## Step 5 — Paper Reading Protocol for This Programme

For every paper in `papers.md` (across all modules), log your Pass-3 in `paper-notes/`:

```
paper-notes/YYYY-Firstname-LastnameEtAl_ShortTitle.md
```

Template:
```
# [Title] ([Year])
**Authors:** ...
**DOI:** ...
**Read:** YYYY-MM-DD
**Pass 1 (skim):** [one sentence: what is this about]
**Pass 2 (read):** [2–3 sentences: what did they do and find]
**Pass 3 (reconstruct):** [did you reconstruct the method unaided? Y/N]
**Questions/fuzzy points:** [...]
**Relevance to my work:** [...]
```

---
## Step 8 — Exercises

See E01 in `exercises/README.md` (paper anatomy analysis of a real bioinformatics paper).

---
## Step 10 — Quiz

1. Name the five parts of the IMRaD structure. What question does each section answer?
2. What is the C-C-C scheme? Apply it to this sentence: "AUROC improved from 0.81 to
   0.92 after feature selection." Rewrite it in C-C-C order.
3. At what reading speed should you decide whether to file, read, or skip a paper?
   What specifically do you look at?
4. What is the Pass-3 protocol? Why is it a better metric of comprehension than
   whether you read the paper?

---
## Step 12 — Reflection

> *[In one paragraph: pick a paper you read recently in this programme. Evaluate
> its structure against the IMRaD checklist. What did the authors do well? Where
> did the argument break down? Would you have organized it differently?]*

---
*Next: `02_writing_abstracts.ipynb`*